<a href="https://colab.research.google.com/github/lucassouza147770-art/MLCB_LG_2026/blob/main/AULA_06/lab01_aula.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Construção do modelo de aprendizado supervisionado
# LAB 03: Expansão de Classe (Data Drift & Novas Intenções)

# BLOCO 1: Preparação do Ambiente e Carga do Dataset

!pip install gensim
!python -m spacy download pt_core_news_sm

import re
import numpy as np
import pandas as pd
import spacy
import gensim.downloader as api
from sklearn.tree import DecisionTreeClassifier
import gradio as gr

# 1. Carregamento do modelo de linguagem em português
print("Carregando modelo morfológico Spacy (pt_core_news_sm)")
nlp = spacy.load("pt_core_news_sm")

# 2. Carregamento dos Word Embeddings
print("Carregando espaço vetorial denso de embeddings modelo Glove")
word_vectors = api.load("glove-wiki-gigaword-50")

# 3. Base de Dados Supervisionada: SAC de Imóveis
dados_imobiliaria = [
    # Intenção: comprar_imovel
    ("Quero comprar um apartamento de 3 quartos com varanda", "comprar_imovel"),
    ("Gostaria de ver casas à venda no centro da cidade", "comprar_imovel"),
    ("Qual o preço médio para compra de cobertura com piscina?", "comprar_imovel"),
    ("Procuro imóvel residencial para comprar com financiamento", "comprar_imovel"),
    ("Vocês têm sobrados à venda na zona sul?", "comprar_imovel"),

    # Intenção: alugar_imovel
    ("Procurando kitnet para alugar perto da faculdade", "alugar_imovel"),
    ("Qual o valor do aluguel deste apartamento de 2 dormitórios?", "alugar_imovel"),
    ("Quero alugar um galpão comercial para minha empresa", "alugar_imovel"),
    ("Quais imóveis estão disponíveis para locação imediata?", "alugar_imovel"),
    ("Preciso de uma casa para alugar que aceite animais", "alugar_imovel"),

    # Intenção: suporte_manutencao
    ("O chuveiro do apartamento alugado queimou, como pedir conserto?", "suporte_manutencao"),
    ("Muro da casa está com infiltração e vazamento de água", "suporte_manutencao"),
    ("Preciso do contato do encanador para reparo na cozinha", "suporte_manutencao"),
    ("A porta da varanda quebrou, quem faz a manutenção?", "suporte_manutencao"),
    ("Vazamento no teto do banheiro precisa de reparo urgente", "suporte_manutencao"),

    # Intenção: 2via_boleto_contrato
    ("Como faço para baixar a segunda via do boleto do aluguel?", "2via_boleto_contrato"),
    ("Não recebi o boleto deste mês para pagamento", "2via_boleto_contrato"),
    ("Preciso do informe de rendimentos e cópia do contrato", "2via_boleto_contrato"),
    ("Onde pego o boleto atualizado com o valor do condomínio?", "2via_boleto_contrato"),
    ("Quero solicitar a segunda via do recibo de pagamento", "2via_boleto_contrato"),

    # =====================================================
    # LAB 03 - Nova Intenção: cancelar_contrato
    # =====================================================
    ("Quero cancelar meu contrato de aluguel", "cancelar_contrato"),
    ("Gostaria de solicitar o cancelamento do contrato", "cancelar_contrato"),
    ("Como faço para cancelar meu contrato?", "cancelar_contrato"),
    ("Quero encerrar o contrato do imóvel", "cancelar_contrato"),
    ("Preciso cancelar o contrato de locação", "cancelar_contrato")
]

df = pd.DataFrame(
    dados_imobiliaria,
    columns=["mensagem", "intencao"]
)

print(
    f"Dataset carregado com {len(df)} mensagens "
    f"divididas em {df['intencao'].nunique()} intenções."
)


# ==========================================================
# BLOCO 2: Esteira NLU
# Pré-processamento e Vetorização
# ==========================================================

def preprocessar_texto(texto: str) -> str:

    texto_limpo = texto.lower()

    texto_limpo = re.sub(
        r'[^a-záàâãéèêíïóôõöúçñ\s]',
        '',
        texto_limpo
    )

    doc = nlp(texto_limpo)

    tokens = [
        token.lemma_
        for token in doc
        if not token.is_stop
        and not token.is_space
        and len(token.text) > 1
    ]

    return " ".join(tokens)


def extrair_sentence_embedding(
    texto_limpo: str,
    modelo_emb
) -> np.ndarray:

    palavras = texto_limpo.split()

    vetores = [
        modelo_emb[p]
        for p in palavras
        if p in modelo_emb
    ]

    if len(vetores) == 0:
        return np.zeros(modelo_emb.vector_size)

    return np.mean(vetores, axis=0)


# Aplicação no Dataset
df['mensagem_limpa'] = df['mensagem'].apply(
    preprocessar_texto
)

X_densos = np.array([
    extrair_sentence_embedding(
        txt,
        word_vectors
    )
    for txt in df['mensagem_limpa']
])

y = df['intencao'].values


# ==========================================================
# BLOCO 3: Treinamento do Modelo e Respostas
# ==========================================================

# Treinamento da Árvore de Decisão
modelo_nlu = DecisionTreeClassifier(
    random_state=42
)

modelo_nlu.fit(
    X_densos,
    y
)

print("Modelo supervisionado treinado!")


# Base de Conhecimento
RESPOSTAS_PADRAO = {

    "comprar_imovel": (
        "**Atendimento de Vendas:** Ficamos felizes com seu interesse! "
        "Você pode conferir nosso catálogo de imóveis à venda "
        "em nosso site www.imobiliaria.com/vendas "
        "ou aguardar que um de nossos corretores entrará em contato em instantes."
    ),

    "alugar_imovel": (
        "**Atendimento de Locação:** Temos ótimas opções disponíveis! "
        "Acesse www.imobiliaria.com/aluguel para filtrar por região e valor. "
        "Para agendar uma visita, envie o código do imóvel por aqui."
    ),

    "suporte_manutencao": (
        "**Suporte e Manutenção:** Sentimos muito pelo inconveniente. "
        "Por favor, abra um chamado urgente em nosso portal do inquilino "
        "(www.imobiliaria.com/manutencao) anexando fotos ou vídeos do problema "
        "para acionarmos nossos prestadores."
    ),

    "2via_boleto_contrato": (
        "**Financeiro e Contratos:** Para acessar boletos ou documentos, "
        "acesse a Área do Cliente em www.imobiliaria.com/cliente "
        "informando seu CPF e senha. Lá você baixa a 2ª via atualizada em segundos."
    ),

    # =====================================================
    # LAB 03 - Resposta da nova intenção
    # =====================================================
    "cancelar_contrato": (
        "**Atendimento de Contratos:** Para solicitar o cancelamento "
        "ou distrato do contrato, entre em contato com nossa equipe "
        "de atendimento para verificar as condições e os procedimentos "
        "necessários para o encerramento do contrato."
    )
}


# ==========================================================
# BLOCO 4: Motor de Inferência com Fallback
# ==========================================================

# Mantido em 65% conforme o LAB 02
LIMIAR_CONFIANCA = 0.65


def processar_atendimento_sac(mensagem_usuario: str):

    if not mensagem_usuario or not mensagem_usuario.strip():

        return (
            "N/A",
            "0.0%",
            "Aguardando mensagem...",
            "Aguardando entrada do usuário..."
        )

    # Step 1: Pré-processamento
    msg_limpa = preprocessar_texto(
        mensagem_usuario
    )

    # Step 2: Vetorização
    vetor_input = extrair_sentence_embedding(
        msg_limpa,
        word_vectors
    ).reshape(1, -1)

    # Step 3: Predição de Probabilidades
    probabilidades = modelo_nlu.predict_proba(
        vetor_input
    )[0]

    idx_maior_prob = np.argmax(
        probabilidades
    )

    confianca = probabilidades[
        idx_maior_prob
    ]

    intencao_detectada = modelo_nlu.classes_[
        idx_maior_prob
    ]

    percentual_confianca = f"{confianca * 100:.1f}%"

    # Step 4: Regra de Decisão / Fallback
    if confianca >= LIMIAR_CONFIANCA:

        classificacao_status = (
            f"IDENTIFICADO ({intencao_detectada}) "
            f"- Corte de confiança: "
            f"{LIMIAR_CONFIANCA * 100:.0f}%"
        )

        texto_resposta = RESPOSTAS_PADRAO[
            intencao_detectada
        ]

    else:

        classificacao_status = (
            f"UNCERTAIN (Fallback Acionado) "
            f"- Confiança abaixo do corte de "
            f"{LIMIAR_CONFIANCA * 100:.0f}%"
        )

        texto_resposta = (
            "Desculpe, não consegui compreender com clareza "
            "a sua solicitação. Estou transferindo agora mesmo "
            "sua conversa para um de nossos atendentes. "
            "Por favor, aguarde um momento."
        )

    # Card visual da resposta
    card_resposta = f"""
    <div style="background-color: #f0f4f9;
                border-left: 5px solid #2b5c8f;
                padding: 15px;
                border-radius: 8px;
                margin-top: 10px;">

        <h4 style="margin: 0 0 8px 0;
                   color: #2b5c8f;">
            Resposta Automática do SAC:
        </h4>

        <p style="margin: 0;
                  font-size: 15px;
                  color: #1a1a1a;">
            {texto_resposta}
        </p>

    </div>
    """

    return (
        intencao_detectada,
        percentual_confianca,
        classificacao_status,
        card_resposta
    )


# ==========================================================
# BLOCO 5: Interface Gradio
# ==========================================================

import gradio as gr

with gr.Blocks(
    theme=gr.themes.Soft(),
    title="SAC Imobiliário"
) as app:

    gr.Markdown(
        """
        # SAC Imobiliário — ChatBot Inteligente

        *Será um prazer atendê-lo. Digite a seguir a sua necessidade*
        """
    )

    with gr.Row():

        # COLUNA DA ESQUERDA
        with gr.Column(scale=1):

            gr.Markdown(
                "### Mensagem do Cliente"
            )

            input_texto = gr.Textbox(
                lines=4,
                placeholder=(
                    "Ex: Preciso da segunda via "
                    "do boleto de aluguel..."
                ),
                label="Digite sua necessidade"
            )

            btn_processar = gr.Button(
                "Processar Mensagem",
                variant="primary",
                size="lg"
            )

            # Exemplos para teste
            gr.Examples(
                examples=[
                    [
                        "Preciso de suporte técnico "
                        "para consertar vazamento."
                    ],
                    [
                        "Quero ver apartamentos "
                        "à venda na zona sul."
                    ],
                    [
                        "Como faço para alugar "
                        "um galpão comercial?"
                    ],
                    [
                        "Gostaria de baixar o "
                        "boleto do condomínio."
                    ],
                    [
                        "Quero cancelar meu contrato "
                        "de aluguel."
                    ],
                    [
                        "Vocês vendem terreno "
                        "na Lua ou em Marte?"
                    ]
                ],
                inputs=input_texto
            )

        # COLUNA DA DIREITA
        with gr.Column(scale=1):

            gr.Markdown(
                "### Painel de Diagnóstico"
            )

            with gr.Row():

                out_intencao = gr.Textbox(
                    label="Intenção",
                    scale=2,
                    interactive=False
                )

                out_confianca = gr.Textbox(
                    label="Confiança",
                    scale=1,
                    interactive=False
                )

            out_status = gr.Textbox(
                label="Status da Decisão",
                interactive=False
            )

            out_resposta = gr.HTML(
                value=(
                    "<div style='padding: 15px; color: #888;'>"
                    "Aguardando envio de mensagem..."
                    "</div>"
                ),
                label="Resposta da Imobiliária"
            )

    # Ação do botão
    btn_processar.click(
        fn=processar_atendimento_sac,
        inputs=[input_texto],
        outputs=[
            out_intencao,
            out_confianca,
            out_status,
            out_resposta
        ]
    )


# ==========================================================
# EXECUÇÃO DA APLICAÇÃO
# ==========================================================

app.launch(
    debug=True,
    share=True
)

# FIM DO CÓDIGO

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 55.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.0/13.0 MB 109.6 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('pt_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Carregando modelo morfológico Spacy (pt_core_news_sm)
Carregando espaço vetorial denso de embeddings modelo Glove
[==================================================] 100.0% 66.0/66.0MB downloaded
Dataset carregado com 25 mensagens divididas em 5 intenções.
Modelo supervisionado treinado!


/tmp/ipykernel_629/2962867844.py:310: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://44fbbbd6872d0f9442.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://44fbbbd6872d0f9442.gradio.live
